## My notes on QuTiP

QuTiP is a python package for calculations and numerical simulations of quantum systems. 

1. You can do calculations with wavefunctions, quantum operators of single and composite systems, and superoperators for open systems
2. It includes solvers for time-evolution: Schrodinger equations, von Neumann, master equations, Floquet formalism, Monte-Carlo quantum trajectors, experimental implementations of the stochastic Schrodinger/master equation

In [8]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from qutip import (Qobj, about, basis, coherent, coherent_dm, create, destroy,
                   expect, fock, fock_dm, mesolve, qeye, sigmax, sigmay,
                   sigmaz, tensor, thermal_dm, anim_matrix_histogram,
                   anim_fock_distribution)
# set a parameter to see animations in line
from matplotlib import rc
rc('animation', html='jshtml')

%matplotlib inline

### Python's Organizational Hierarchy (From smallest to largest)


1. Object
An object is the most basic unit in Python. Everything in Python is an object (integers, strings, etc.). It is a specific "thing" that exists in your computer's memory while the program is running.

Code Analogy: A single variable or a specific instance of a class.

2. Class
A Class is the blueprint or template used to create objects. it defines what data the object will hold and what it can do.

Code Analogy: class Dog:. It’s not a dog yet; it’s the definition of what a dog is.

3. Module
A Module is simply a single Python file (.py). It’s a way to group related classes, functions, and variables together so they don't clutter a single file.

Code Analogy: utils.py or models.py.

4. Package
A Package is a collection of modules organized in a directory (folder). It allows you to group multiple modules together under a single name.

Code Analogy: A folder named analytics/ containing __init__.py, charts.py, and tables.py.

5. Library
A Library is a high-level collection of packages and modules designed to solve a specific problem (e.g., data science, web scraping). It is the "finished product" you install.

Code Analogy: pandas, requests, or scikit-learn.

Imagine you are building a Racing Game:

Object: red_ferrari (A specific car currently at position X, Y).

Class: Car (The code that defines that all cars have wheels and can "drive").

Module: physics.py (A file containing all the math for gravity and collisions).

Package: game_engine/ (A folder containing physics.py, graphics.py, and sound.py).

Library: PyGame (The entire external toolkit you downloaded to make the game possible).

#### Qobj class
is used to represent quantum objects such as states and operators. The Qobj class contains all the information required to describe a quantum system, such as its matrix representation, composite structure and dimensionality.

It’s the code that says: "Every quantum object must have a matrix, a shape, and the ability to perform operations like 'transpose' or 'dagger'."

In [6]:
import qutip as qt

# We aren't creating a specific state yet; 
# we are just looking at the 'Qobj' class definition.
# Inside QuTiP's source code, it looks roughly like this:
class Qobj2:
    def __init__(self, data):
        self.data = data  # The matrix/vector
        self.dims = ...   # The dimensions
        
    def dagger(self):
        # Logic to return the conjugate transpose
        pass

##### The Object

When you actually create a specific state—like a "Spin Up" state or a "Schrödinger's Cat" state—you are creating an Object (also called an Instance) of that class. 

We can create a new quantum object using the Qobj class constructor, like this:

In [9]:
q = Qobj([[1], [0]])

q

Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]

Here we passed python list as an argument to the class constructor. The data in this list is used to construct the matrix representation of the quantum objects, and the other properties of the quantum object is by default computed from the same data.

Another way to do this: 

In [10]:
# Creating an 'Object' named 'psi'
# psi is an instance of the Qobj class
psi = qt.basis(2, 0) 

# psi now has "State" (its data)
print(psi.data) # Output: The vector [1, 0]

# psi now has "Behavior" (its methods)
psi_bra = psi.dag() # Uses the 'dagger' method defined in the class

Dense(shape=(2, 1), fortran=True)


We can inspect the properties of a Qobj instance using the following class method:

In [15]:
# the dimension, or composite Hilbert state space structure
print(q.dims)
# the shape of the matrix data representation
print(q.shape)
# the matrix data itself. in sparse matrix format.
print(q.data)
# get the dense matrix representation
print(q.full())
# some additional properties
q.isherm, q.type

[[2], [1]]
(2, 1)
Dense(shape=(2, 1), fortran=True)
[[1.+0.j]
 [0.+0.j]]


(False, 'ket')

#### Qobj Class Attributes vs. Methods

In Python classes, we distinguish between what the object is (Attributes) and what the object does (Methods).

Attributes (The "Is")
These are variables attached to the object. In QuTiP, a Qobj has attributes that describe its physical properties.

psi.type: Tells you if it's a ket, bra, or operator.

psi.dims: Tells you the Hilbert space structure.

Methods (The "Does")
These are functions attached to the object. They perform operations on the data within the object.

psi.norm(): Calculates the length of the vector.

psi.expt_value(operator): Calculates the expectation value.

Imagine if QuTiP didn't use classes. You would have to pass your matrix into every single function manually:
qt.calculate_norm(my_matrix)

With Classes, the data and the functions live together. Because psi is a Qobj, it "knows" how to normalize itself:
psi.norm()

### States and operators

Normally we do not need to create Qobj instances from stratch, using its constructor and passing its matrix represantation as argument. Instead we can use functions in QuTiP that generates common states and operators for us. Here are some examples of built-in state functions:

In [17]:
# Fundamental basis states (Fock states of oscillator modes)

N = 3  # number of states in the Hilbert space
n = 1  # the state that will be occupied

basis(N, n)  # equivalent to fock(N, n)

Quantum object: dims=[[3], [1]], shape=(3, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]
 [0.]]

In [18]:
fock(4, 2)  # another example

Quantum object: dims=[[4], [1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [0.]
 [1.]
 [0.]]

In [19]:
# a coherent state
coherent(N=10, alpha=1.0)

Quantum object: dims=[[10], [1]], shape=(10, 1), type='ket', dtype=Dense
Qobj data =
[[0.60653066]
 [0.60653066]
 [0.42888194]
 [0.24761511]
 [0.12380753]
 [0.0553686 ]
 [0.02260303]
 [0.00854887]
 [0.00299672]
 [0.00110007]]

##### Density matrix

In [20]:
# a fock state as density matrix
fock_dm(5, 2)  # 5 = hilbert space size, 2 = state that is occupied

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

#### Operators
Qubit (two-level system) operators

In [23]:
# Pauli sigma x
sigmax()

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 1.]
 [1. 0.]]

Harmonic oscillator operators

In [24]:
#  annihilation operator

destroy(N=5)  # N = number of fock states included in the Hilbert space

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=False
Qobj data =
[[0.         1.         0.         0.         0.        ]
 [0.         0.         1.41421356 0.         0.        ]
 [0.         0.         0.         1.73205081 0.        ]
 [0.         0.         0.         0.         2.        ]
 [0.         0.         0.         0.         0.        ]]

##### **Creation Operator ($a^\dagger$)**
The creation operator $a^\dagger$ acting on a Fock state $|n\rangle$ is defined as:
$$a^\dagger |n\rangle = \sqrt{n+1} |n+1\rangle$$

Because of this, the matrix elements are not just $1, 2, 3, \dots$, but rather the square roots of the integers: $\sqrt{1}, \sqrt{2}, \sqrt{3}, \dots$. In the matrix representation, these values populate the **first sub-diagonal** (below the main diagonal).



##### **Annihilation Operator ($a$)**
The annihilation operator $a$ acting on a Fock state $|n\rangle$ is defined as:
$$a |n\rangle = \sqrt{n} |n-1\rangle$$

Similarly, the matrix elements are the square roots of the integers: $\sqrt{1}, \sqrt{2}, \sqrt{3}, \dots$. However, in the matrix representation, these values populate the **first super-diagonal** (above the main diagonal).

### Composite systems
In most cases we are interested in coupled quantum systems, for example coupled qubits, a qubit coupled to a cavity (oscillator mode), etc.

To define states and operators for such systems in QuTiP, we use the tensor function to create Qobj instances for the composite system.

For example, consider a system composed of two qubits. If we want to create a Pauli  σz
  operator that acts on the first qubit and leaves the second qubit unaffected (i.e., the operator  σz⊗1 ), we would do:

In [25]:
sz1 = tensor(sigmaz(), qeye(2))

sz1

Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 1.  0.  0.  0.]
 [ 0.  1.  0.  0.]
 [ 0.  0. -1.  0.]
 [ 0.  0.  0. -1.]]

### Unitary Dynamics

In quantum mechanics, **Unitary Dynamics** refers to how a quantum state evolves over time while preserving its total probability (the "length" of the vector). 

In the hierarchy of Python, QuTiP handles this using specific **Functions** and **Classes** designed to solve the Schrödinger equation.


##### 1. The Core Concept: Preservation
A transformation $U$ is **Unitary** if it satisfies $U^\dagger U = I$. 
* **The Goal:** If you start with a state $|\psi(0)\rangle$, you want to find $|\psi(t)\rangle$.
* **The Math:** $|\psi(t)\rangle = U(t) |\psi(0)\rangle$.
* **The Physics:** This ensures that if the sum of probabilities was 100% at the start, it stays 100% forever.

---

##### 2. Unitary Evolution in QuTiP: `sesolve`
To simulate this, QuTiP provides a powerful function in the `solver` module called `sesolve` (Schrödinger Equation Solver).

###### The Hierarchy Breakdown:
* **Module:** `qutip.solver`
* **Function:** `sesolve()`
* **Input Objects:** A Hamiltonian $H$ (Qobj), an initial state $\psi_0$ (Qobj), and a list of times $t$.
* **Output Object:** A `Result` object (a custom Class that holds the state at every time step).

---

##### 3. Code Example: A Rotating Spin
Imagine a spin-1/2 particle in a magnetic field. The Hamiltonian $H = \frac{1}{2}\sigma_z$ causes the state to rotate.

In [29]:
import qutip as qt
import numpy as np

# 1. Define the Objects (Hamiltonian and Initial State)
H = 0.5 * qt.sigmaz() 
psi0 = (qt.basis(2, 0) + qt.basis(2, 1)).unit() # State pointing along X

# 2. Define the Time Range
times = np.linspace(0, 10, 100)

# 3. Run the Unitary Dynamics
# result is an 'Instance' of the Result class
result = qt.sesolve(H, psi0, times)

# 4. Accessing the Result Methods/Attributes
final_state = result.states[-1] 

print(H,psi0, times)

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 0.5  0. ]
 [ 0.  -0.5]] Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678]
 [0.70710678]] [ 0.          0.1010101   0.2020202   0.3030303   0.4040404   0.50505051
  0.60606061  0.70707071  0.80808081  0.90909091  1.01010101  1.11111111
  1.21212121  1.31313131  1.41414141  1.51515152  1.61616162  1.71717172
  1.81818182  1.91919192  2.02020202  2.12121212  2.22222222  2.32323232
  2.42424242  2.52525253  2.62626263  2.72727273  2.82828283  2.92929293
  3.03030303  3.13131313  3.23232323  3.33333333  3.43434343  3.53535354
  3.63636364  3.73737374  3.83838384  3.93939394  4.04040404  4.14141414
  4.24242424  4.34343434  4.44444444  4.54545455  4.64646465  4.74747475
  4.84848485  4.94949495  5.05050505  5.15151515  5.25252525  5.35353535
  5.45454545  5.55555556  5.65656566  5.75757576  5.85858586  5.95959596
  6.06060606  6.16161616  6.26262626

##### 4. Why is it "Unitary"?
If you look at the **Module** logic inside QuTiP, `sesolve` is essentially calculating the **Unitary Operator** $U(t) = e^{-iHt}$.

Because $H$ is **Hermitian** ($H = H^\dagger$), the matrix exponentiation $e^{-iHt}$ is guaranteed to be **Unitary**. In QuTiP, you can even create this operator directly using a method on the Hamiltonian object:

In [31]:
# Create the Unitary Operator Object directly
t=1
U = (-1j * H * t).expm() 
psi_t = U * psi0

###### 5. Summary Table

| Dynamics Component | Python/QuTiP Element | Role |
| :--- | :--- | :--- |
| **Hamiltonian ($H$)** | `Qobj` (Object) | Defines the "energy" and the rules of motion. |
| **Evolution ($U$)** | `sesolve` (Function) | The engine that moves the state through time. |
| **Output Data** | `Result` (Class) | A container that stores all $\psi(t)$ and expectation values. |
| **Probability** | `.norm()` (Method) | Used to verify the state stays at 1.0 (Unitary check). |

### Dissipative Dynamics

Once you add **dissipation** (noise, friction, or decoherence), you transition from **Unitary Dynamics** to **Open Quantum Systems**. The physics shifts from the Schrödinger Equation to the **Lindblad Master Equation**.

In the Python hierarchy, this move changes the "Object" we use to represent the state and the "Function" we use to solve the evolution.

###### 1. From State Vectors to Density Matrices
In unitary dynamics, we use a **Ket** ($|\psi\rangle$). In dissipative dynamics, we must use a **Density Matrix** ($\rho$).
* **Ket:** Represents a "pure" state.
* **Density Matrix:** Represents a "mixed" state (a statistical ensemble).
* **QuTiP Logic:** You can turn a ket into a density matrix using the `.ptrace()` method or the `qt.ket2dm()` function.


###### 2. The "Collapse" Operators (The Dissipation Logic)
To tell QuTiP *how* the system is losing energy or coherence, we define **Collapse Operators** ($C_n$). These are standard `Qobj` operators that represent the interaction with the environment.

Common examples for a Harmonic Oscillator:
* **Photon Loss (Decay):** `sqrt(gamma) * qt.destroy(N)`
* **Dephasing:** `sqrt(gamma) * qt.num(N)`


###### 3. The Solver: `mesolve`
Instead of `sesolve` (Schrödinger), we now use **`mesolve`** (Master Equation).


In [ ]:
import qutip as qt
import numpy as np

N = 10
gamma = 0.1  # Decay rate

# 1. Setup the Objects
H = qt.num(N)                # Simple Hamiltonian
psi0 = qt.basis(N, 9)        # Start in the 9th excited state
collapse_ops = [np.sqrt(gamma) * qt.destroy(N)] # Loss of one photon

times = np.linspace(0, 20, 100)

# 2. Run the Dissipative Dynamics
# mesolve automatically handles the Lindblad math
result = qt.mesolve(H, psi0, times, collapse_ops, [qt.num(N)])

# 3. Accessing the results
# Because we passed [qt.num(N)] as the 5th argument, 
# result.expect[0] contains the population over time.


##### 4. How the Hierarchy Evolves

| Feature | Unitary (`sesolve`) | Dissipative (`mesolve`) |
| :--- | :--- | :--- |
| **System Type** | Closed (Isolated) | Open (Connected to environment) |
| **State Object** | Ket ($|\psi\rangle$) | Density Matrix ($\rho$) |
| **Evolution** | Reversible (Unitary) | Irreversible (Arrow of time) |
| **Required Inputs** | $H$ | $H$ + `c_ops` (List of collapse objects) |


###### 5. Why do we need this? (Code Analogy)
Think of **Unitary Dynamics** like a **Pendulum in a Vacuum**. It swings back and forth forever ($U^\dagger U = I$).

Think of **Dissipation** like **Air Resistance**. The pendulum eventually slows down and stops at the bottom. 
* In QuTiP, the `collapse_ops` act like the "friction" data that tells the `mesolve` function how to drain energy out of your `Qobj` state over time.